# Processed E-commerce Dataset

This notebook creates a clean processed e-commerce dataset from three input CSV files:

- **Orders** — transaction-level order information
- **Customers** — customer and membership information
- **Products** — product, category, brand, and pricing information

The workflow demonstrates `pandas`, `merge()`, `concat()`, `apply()`, and DateTime operations, then exports the final dataset as a CSV file.

## 1. Import Libraries

In [14]:
import pandas as pd

## 2. Load the Input Datasets

In [15]:
# Keep the three CSV files in the same folder as this notebook.
customers = pd.read_csv("/Day9_Customers.csv")
orders = pd.read_csv("/Day9_Orders.csv")
products = pd.read_csv("/Day9_Products.csv")

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Products:", products.shape)

display(customers.head())
display(orders.head())
display(products.head())

Customers: (30, 5)
Orders: (120, 7)
Products: (20, 5)


,Customer_ID,Customer_Name,City,Region,Membership_Type
0,C001,Aarav Sharma,Srinagar,North,Premium
1,C002,Zoya Khan,Delhi,North,Regular
2,C003,Rohan Mehta,Mumbai,West,Premium
3,C004,Ananya Singh,Jammu,North,Regular
4,C005,Kabir Ali,Lucknow,North,New


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered


,Product_ID,Product_Name,Category,Unit_Price,Brand
0,P001,Wireless Headphones,Electronics,1499,SoundMax
1,P002,Mechanical Keyboard,Electronics,2499,KeyPro
2,P003,Wireless Mouse,Electronics,899,TechGear
3,P004,Smart Watch,Electronics,3299,FitTech
4,P005,Power Bank,Electronics,1199,VoltPlus


## 3. Demonstrate `concat()`

In [16]:
# Split the Orders DataFrame into two parts and combine them back using concat().
orders_part1 = orders.iloc[:60].copy()
orders_part2 = orders.iloc[60:].copy()

orders_combined = pd.concat([orders_part1, orders_part2], ignore_index=True)

print("First part:", orders_part1.shape)
print("Second part:", orders_part2.shape)
print("After concat:", orders_combined.shape)

display(orders_combined.head())

First part: (60, 7)
Second part: (60, 7)
After concat: (120, 7)


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered


## 4. Merge Orders with Customer and Product Information

In [17]:
# Merge customer information using Customer_ID.
processed = orders_combined.merge(
    customers,
    on="Customer_ID",
    how="left"
)

# Merge product information using Product_ID.
processed = processed.merge(
    products,
    on="Product_ID",
    how="left"
)

print("Shape after merges:", processed.shape)
display(processed.head())

Shape after merges: (120, 15)


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status,Customer_Name,City,Region,Membership_Type,Product_Name,Category,Unit_Price,Brand
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered,Harsh Vardhan,Noida,North,Premium,Cricket Bat,Sports,2499,BatPro
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered,Ishita Gupta,Bengaluru,South,Premium,Wireless Mouse,Electronics,899,TechGear
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered,Karan Joshi,Chandigarh,North,Regular,Smart Watch,Electronics,3299,FitTech
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered,Maryam Khan,Hyderabad,South,Regular,Machine Learning Basics,Books,999,AIPress
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered,Reyansh Jain,Kolkata,East,New,Coffee Maker,Home & Kitchen,3499,HomeBrew


## 5. DateTime Operations

In [18]:
# Convert Order_Date to a proper datetime column.
processed["Order_Date"] = pd.to_datetime(processed["Order_Date"])

# Extract useful date information.
processed["Month"] = processed["Order_Date"].dt.month
processed["Month_Name"] = processed["Order_Date"].dt.month_name()
processed["Day"] = processed["Order_Date"].dt.day
processed["Day_of_Week"] = processed["Order_Date"].dt.day_name()
processed["Is_Weekend"] = processed["Order_Date"].dt.dayofweek >= 5

display(processed[[
    "Order_ID", "Order_Date", "Month", "Month_Name",
    "Day", "Day_of_Week", "Is_Weekend"
]].head())

,Order_ID,Order_Date,Month,Month_Name,Day,Day_of_Week,Is_Weekend
0,O0001,2026-02-19,2,February,19,Thursday,False
1,O0002,2026-01-25,1,January,25,Sunday,True
2,O0003,2026-02-26,2,February,26,Thursday,False
3,O0004,2026-03-04,3,March,4,Wednesday,False
4,O0005,2026-03-29,3,March,29,Sunday,True


## 6. Create Useful Columns with `apply()`

In [19]:
# Calculate the total value of each order.
processed["Total_Amount"] = processed.apply(
    lambda row: row["Quantity"] * row["Unit_Price"],
    axis=1
)

# Categorize order quantities.
processed["Quantity_Category"] = processed["Quantity"].apply(
    lambda q: "High" if q >= 4 else ("Medium" if q >= 2 else "Low")
)

# Create a readable customer label combining name and membership type.
processed["Customer_Label"] = processed.apply(
    lambda row: f"{row['Customer_Name']} ({row['Membership_Type']})",
    axis=1
)

display(processed[[
    "Order_ID", "Customer_Label", "Quantity",
    "Quantity_Category", "Unit_Price", "Total_Amount"
]].head())

,Order_ID,Customer_Label,Quantity,Quantity_Category,Unit_Price,Total_Amount
0,O0001,Harsh Vardhan (Premium),2,Medium,2499,4998
1,O0002,Ishita Gupta (Premium),2,Medium,899,1798
2,O0003,Karan Joshi (Regular),1,Low,3299,3299
3,O0004,Maryam Khan (Regular),3,Medium,999,2997
4,O0005,Reyansh Jain (New),5,High,3499,17495


## 7. Organize the Final DataFrame

In [20]:
# Arrange the columns in a clean, meaningful order.
final_columns = [
    "Order_ID", "Order_Date", "Month", "Month_Name", "Day",
    "Day_of_Week", "Is_Weekend", "Customer_ID", "Customer_Name",
    "Customer_Label", "City", "Region", "Membership_Type",
    "Product_ID", "Product_Name", "Category", "Brand", "Quantity",
    "Quantity_Category", "Unit_Price", "Total_Amount",
    "Payment_Method", "Order_Status"
]

processed = processed[final_columns]

print("Final shape:", processed.shape)
print("Missing values:")
display(processed.isna().sum())
display(processed.head(10))

Final shape: (120, 23)
Missing values:


,0
Order_ID,0
Order_Date,0
Month,0
Month_Name,0
Day,0
Day_of_Week,0
Is_Weekend,0
Customer_ID,0
Customer_Name,0
Customer_Label,0


,Order_ID,Order_Date,Month,Month_Name,Day,Day_of_Week,Is_Weekend,Customer_ID,Customer_Name,Customer_Label,...,Product_ID,Product_Name,Category,Brand,Quantity,Quantity_Category,Unit_Price,Total_Amount,Payment_Method,Order_Status
0,O0001,2026-02-19,2,February,19,Thursday,False,C027,Harsh Vardhan,Harsh Vardhan (Premium),...,P019,Cricket Bat,Sports,BatPro,2,Medium,2499,4998,Credit Card,Delivered
1,O0002,2026-01-25,1,January,25,Sunday,True,C006,Ishita Gupta,Ishita Gupta (Premium),...,P003,Wireless Mouse,Electronics,TechGear,2,Medium,899,1798,Debit Card,Delivered
2,O0003,2026-02-26,2,February,26,Thursday,False,C015,Karan Joshi,Karan Joshi (Regular),...,P004,Smart Watch,Electronics,FitTech,1,Low,3299,3299,Cash on Delivery,Delivered
3,O0004,2026-03-04,3,March,4,Wednesday,False,C024,Maryam Khan,Maryam Khan (Regular),...,P015,Machine Learning Basics,Books,AIPress,3,Medium,999,2997,Net Banking,Delivered
4,O0005,2026-03-29,3,March,29,Sunday,True,C025,Reyansh Jain,Reyansh Jain (New),...,P009,Coffee Maker,Home & Kitchen,HomeBrew,5,High,3499,17495,Credit Card,Delivered
5,O0006,2026-02-09,2,February,9,Monday,False,C003,Rohan Mehta,Rohan Mehta (Premium),...,P018,Football,Sports,SportZone,3,Medium,799,2397,UPI,Delivered
6,O0007,2026-02-10,2,February,10,Tuesday,False,C011,Vivaan Kapoor,Vivaan Kapoor (New),...,P003,Wireless Mouse,Electronics,TechGear,3,Medium,899,2697,UPI,Delivered
7,O0008,2026-03-27,3,March,27,Friday,False,C020,Priya Menon,Priya Menon (Premium),...,P005,Power Bank,Electronics,VoltPlus,4,High,1199,4796,Debit Card,Delivered
8,O0009,2026-03-13,3,March,13,Friday,False,C024,Maryam Khan,Maryam Khan (Regular),...,P009,Coffee Maker,Home & Kitchen,HomeBrew,2,Medium,3499,6998,UPI,Cancelled
9,O0010,2026-03-05,3,March,5,Thursday,False,C026,Fatima Noor,Fatima Noor (Regular),...,P005,Power Bank,Electronics,VoltPlus,2,Medium,1199,2398,Debit Card,Shipped


## 8. Export the Processed Dataset

In [21]:
# Export the final processed dataset.
output_file = "Processed_Ecommerce_Dataset.csv"
processed.to_csv(output_file, index=False)

print(f"Processed dataset exported successfully as: {output_file}")

Processed dataset exported successfully as: Processed_Ecommerce_Dataset.csv


## Result

The final dataset combines order, customer, and product information and includes:

- Merged customer and product details
- DateTime-derived month, day, weekday, and weekend fields
- Order-level `Total_Amount`
- Quantity categories
- A combined customer label
- Clean, organized columns

The output file is **`Processed_Ecommerce_Dataset.csv`**.

In [23]:
#Download file
from google.colab import files

files.download("Processed_Ecommerce_Dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>